# Walkthrough — candidates × datasets

This notebook reads the bucket tables and summary CSVs produced by
`Workflow/visualise.py` (the third step of the pipeline) and renders
the headline visualisations.

Pre-requisite:

```bash
PYTHONPATH=. python Workflow/run_all.py \
    --candidate Candidates/dummy-candidate/candidate.py \
    --dataset kaiser
```

The metric used is **metrik-4** from
[`domain-model-metrics`](https://github.com/VasiliySeibert/domainModel-Metrics-Comparison).
Score buckets are `[0, 0.1)` / `[0.1, 0.2)` / `[0.2, 0.3)` / `[0.3, 1.0]`.
Rows in every table are one per candidate (one candidate per `Workflow/run_all.py` invocation).

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path('Workflow/Results')
assert RESULTS.is_dir(), 'Run Workflow/run_all.py first'

summary = pd.read_csv(RESULTS / '_summary.csv')
errors  = pd.read_csv(RESULTS / '_errors.csv') if (RESULTS / '_errors.csv').exists() else None
print(f'{len(summary)} (metric, candidate, dataset, element) rows')
print(f'{len(errors) if errors is not None else 0} failure rows')
summary.head()

## 1. Bucket tables (the headline deliverable)

One CSV per `(dataset, element, metric)`. Filenames are
    Workflow/Results/_bucket_<dataset>_<element>_<metric>.csv

Each row is one candidate. Columns are bucket counts
("[0, 0.1)", "[0.1, 0.2)", "[0.2, 0.3)", "[0.3, 1.0]") plus `n`,
`n_failed`, `mean`, `median`. Read it as: *"of the N records this
candidate produced, K1 fell in bucket 1, K2 in bucket 2, ..."*.

In [ ]:
bucket_files = sorted(RESULTS.glob('_bucket_*.csv'))
for bf in bucket_files:
    print(f'\n=== {bf.name} ===')
    display(pd.read_csv(bf))

## 2. Heatmap of bucket distribution (Class element)

Visual version of the bucket table: each row is a candidate,
each column is a bucket. The intensity in cell (r, c) shows how many
of the N records for that candidate landed in bucket c.

In [ ]:
def plot_bucket_heatmap(dataset: str, element: str, metric: str):
    f = RESULTS / f'_bucket_{dataset}_{element}_{metric}.csv'
    if not f.exists():
        print(f'  (skip: {f.name} not found)')
        return
    df = pd.read_csv(f)
    bucket_cols = ['[0, 0.1)', '[0.1, 0.2)', '[0.2, 0.3)', '[0.3, 1.0]']
    df['row'] = df['candidate']
    matrix = df.set_index('row')[bucket_cols]

    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(matrix))))
    im = ax.imshow(matrix.values, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(bucket_cols)))
    ax.set_xticklabels(bucket_cols, rotation=20, ha='right')
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix.values[i, j]
            if v:
                ax.text(j, i, str(v), ha='center', va='center', fontsize=8, color='black')
    ax.set_title(f'{dataset} — {element} ({metric}) — records per bucket')
    plt.colorbar(im, ax=ax, label='count')
    plt.tight_layout()
    plt.show()

metrics = sorted(summary['metric'].unique())
for m in metrics:
    for ds in ('kaiser', 'reference'):
        plot_bucket_heatmap(ds, 'class_score', m)

## 3. Mean score per candidate

Headline single-number comparison. Higher is better. Same colour
scheme as the bucket heatmap (yellow = best, red = worst).

In [ ]:
def plot_mean_bars(dataset: str, metric: str):
    sub = summary[(summary['dataset'] == dataset) & (summary['metric'] == metric)]
    if sub.empty:
        print(f'  (skip: no rows for {dataset}, {metric})')
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, max(3, 0.4 * sub['candidate'].nunique())), sharey=False)
    for ax, element in zip(axes, ['class_score', 'attribute_score', 'association_score']):
        pivot = (sub[sub['element'] == element]
                 .pivot_table(index='candidate', values='mean'))
        im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            v = pivot.values[i, 0]
            if pd.notna(v):
                ax.text(0, i, f'{v:.2f}', ha='center', va='center', fontsize=8)
        ax.set_title(element)
    fig.suptitle(f'{dataset} — mean {metric} score per candidate')
    plt.tight_layout()
    plt.show()

for m in sorted(summary['metric'].unique()):
    for ds in ('kaiser', 'reference'):
        plot_mean_bars(ds, m)

## 4. Failure rate per candidate

How often each candidate produced an empty or unparseable PlantUML
block. Useful for spotting unreliable candidates.

In [ ]:
if errors is not None and len(errors) > 0:
    fr = errors.groupby(['dataset', 'candidate']).size().reset_index(name='fail_count')
    n_per = summary.groupby(['dataset', 'candidate'])['n'].first().reset_index()
    fr = fr.merge(n_per, on=['dataset', 'candidate'], how='left')
    fr['fail_pct'] = fr['fail_count'] / fr['n'] * 100
    for ds, sub in fr.groupby('dataset'):
        fig, ax = plt.subplots(figsize=(8, max(2, 0.4 * len(sub))))
        ax.barh(sub['candidate'], sub['fail_pct'], color='salmon')
        ax.set_xlabel('failure rate %')
        ax.set_title(f'{ds} — failure rate %')
        plt.tight_layout()
        plt.show()
else:
    print('No failures recorded.')

## 5. Error log

Every record that produced an error. Useful for debugging prompt
issues — `raw_excerpt` shows the first 200 chars of the candidate's output.

In [ ]:
if errors is not None and len(errors) > 0:
    print(f'{len(errors)} failure rows')
    display(errors.head(30))
else:
    print('No failures recorded.')

## 6. Cross-candidate summary

The flat `_summary.csv` — every `(candidate, dataset, element)`
row. This is the canonical machine-readable artefact. Useful for
spreadsheet pivot tables, statistical tests, or feeding into a
follow-up analysis notebook.

In [ ]:
summary_sorted = summary.sort_values(
    ['dataset', 'element', 'mean'], ascending=[True, True, False]
)
summary_sorted